In [6]:
import pandas as pd
import os
import re
import glob

RASPAGEM_DIR = "Raspagem"
OUTPUT_CSV = os.path.join("dados_tratados", "dados_raspagem_consolidado.csv")

In [7]:
# ── Exploração: estrutura de um arquivo de exemplo ──────────────────────────
files = sorted(glob.glob(os.path.join(RASPAGEM_DIR, "*.xlsx")))
print(f"Total de arquivos encontrados: {len(files)}")

exemplo = pd.read_excel(files[-1])
print(f"\nArquivo de exemplo: {os.path.basename(files[-1])}")
print(f"Shape: {exemplo.shape}")
print(f"\nColunas ({len(exemplo.columns)}):")
for col in exemplo.columns:
    print(f"  {col:35s}  dtype: {exemplo[col].dtype}")
print(f"\nAmostra (3 linhas):")
exemplo.head(3)

Total de arquivos encontrados: 88

Arquivo de exemplo: dados_aedes_20260320_weekid481_1133mosquitos.xlsx
Shape: (910, 23)

Colunas (23):
  id                                   dtype: int64
  week_id                              dtype: int64
  trap_id                              dtype: int64
  agent_id                             dtype: int64
  week                                 dtype: str
  agent                                dtype: str
  status                               dtype: str
  status_id                            dtype: float64
  inspection_id                        dtype: float64
  aedes_aegypti_femea                  dtype: int64
  aedes_aegypti_macho                  dtype: int64
  aedes_albopictus_femea               dtype: int64
  aedes_albopictus_macho               dtype: int64
  culex_sp_femea                       dtype: int64
  culex_sp_macho                       dtype: int64
  total_mosquitos                      dtype: int64
  latitude                       

,id,week_id,trap_id,agent_id,week,agent,status,status_id,inspection_id,aedes_aegypti_femea,...,culex_sp_femea,culex_sp_macho,total_mosquitos,latitude,longitude,street,number,block,complement,neighborhood
0,4775306,481,11107,883,11/2026,Nilvo Masulini de Oliveira,Realizada,10.0,2035697.0,1,...,0,0,1,-30.070737,-51.186928,AV CEL APARICIO BORGES,2001,2237369.0,Colegio Tiradentes,APARICIO BORGES
1,4775307,481,11110,883,11/2026,Nilvo Masulini de Oliveira,Realizada,10.0,2034623.0,1,...,0,0,1,-30.073131,-51.189766,R QUERUBIM COSTA,18,2237288.0,NaN,APARICIO BORGES
2,4780010,481,11096,1065,11/2026,Lucas Bonesso,Realizada,10.0,2033020.0,4,...,0,0,4,-30.069557,-51.184382,AV SILVADO,456,NaN,ESCOLA TIO CHICO,APARICIO BORGES


In [8]:
# ── Carrega todos os arquivos e extrai metadados do nome ─────────────────────
# Padrão do nome: dados_aedes_YYYYMMDD_weekidNNN_Nmosquitos[_sufixo].xlsx
PATTERN = re.compile(r"dados_aedes_(\d{8})_weekid(\d+)_(\d+)mosquitos")

registros = []
for f in files:
    basename = os.path.basename(f)
    m = PATTERN.search(basename)
    if not m:
        print(f"  [AVISO] Arquivo ignorado (nome fora do padrão): {basename}")
        continue
    data_str, week_id, n_mosquitos = m.group(1), int(m.group(2)), int(m.group(3))
    registros.append({
        "arquivo": f,
        "basename": basename,
        "data": pd.to_datetime(data_str, format="%Y%m%d"),
        "week_id": week_id,
        "n_mosquitos_nome": n_mosquitos,
    })

meta = pd.DataFrame(registros).sort_values(["week_id", "n_mosquitos_nome", "data"])
print(f"Arquivos parseados: {len(meta)}")
print(f"\nWeek IDs únicas: {meta['week_id'].nunique()}")
print(f"\nArquivos por week_id (top 10 com mais duplicatas):")
contagem = meta.groupby("week_id").size().sort_values(ascending=False)
print(contagem.head(10).to_string())
meta.head(10)

Arquivos parseados: 88

Week IDs únicas: 31

Arquivos por week_id (top 10 com mais duplicatas):
week_id
470    7
468    7
469    7
480    7
475    6
479    6
474    5
473    5
478    5
481    5


,arquivo,basename,data,week_id,n_mosquitos_nome
0,Raspagem\dados_aedes_20250816_weekid450_62mosq...,dados_aedes_20250816_weekid450_62mosquitos.xlsx,2025-08-16,450,62
1,Raspagem\dados_aedes_20250822_weekid451_42mosq...,dados_aedes_20250822_weekid451_42mosquitos.xlsx,2025-08-22,451,42
2,Raspagem\dados_aedes_20250827_weekid452_92mosq...,dados_aedes_20250827_weekid452_92mosquitos.xlsx,2025-08-27,452,92
3,Raspagem\dados_aedes_20250903_weekid453_45mosq...,dados_aedes_20250903_weekid453_45mosquitos.xlsx,2025-09-03,453,45
4,Raspagem\dados_aedes_20250913_weekid454_74mosq...,dados_aedes_20250913_weekid454_74mosquitos.xlsx,2025-09-13,454,74
5,Raspagem\dados_aedes_20250920_weekid455_158mos...,dados_aedes_20250920_weekid455_158mosquitos.xlsx,2025-09-20,455,158
6,Raspagem\dados_aedes_20250927_weekid456_193mos...,dados_aedes_20250927_weekid456_193mosquitos.xlsx,2025-09-27,456,193
7,Raspagem\dados_aedes_20251004_weekid457_157mos...,dados_aedes_20251004_weekid457_157mosquitos.xlsx,2025-10-04,457,157
8,Raspagem\dados_aedes_20251016_weekid459_306mos...,dados_aedes_20251016_weekid459_306mosquitos.xlsx,2025-10-16,459,306
9,Raspagem\dados_aedes_20251022_weekid460_203mos...,dados_aedes_20251022_weekid460_203mosquitos.xlsx,2025-10-22,460,203


In [9]:
# ── Para cada week_id, seleciona o arquivo com mais mosquitos ─────────────────
# Em caso de empate no nº de mosquitos, usa o arquivo mais recente (data maior).
melhor = (
    meta
    .sort_values(["week_id", "n_mosquitos_nome", "data"], ascending=[True, True, True])
    .groupby("week_id", as_index=False)
    .last()   # último = maior n_mosquitos; em empate, data mais recente
)

print("Week IDs selecionadas:", len(melhor))
print(f"Arquivos descartados: {len(meta) - len(melhor)}")
print("\nArquivos selecionados por week_id:")
print(melhor[["week_id", "data", "n_mosquitos_nome", "basename"]].to_string(index=False))

Week IDs selecionadas: 31
Arquivos descartados: 57

Arquivos selecionados por week_id:
 week_id       data  n_mosquitos_nome                                            basename
     450 2025-08-16                62     dados_aedes_20250816_weekid450_62mosquitos.xlsx
     451 2025-08-22                42     dados_aedes_20250822_weekid451_42mosquitos.xlsx
     452 2025-08-27                92     dados_aedes_20250827_weekid452_92mosquitos.xlsx
     453 2025-09-03                45     dados_aedes_20250903_weekid453_45mosquitos.xlsx
     454 2025-09-13                74     dados_aedes_20250913_weekid454_74mosquitos.xlsx
     455 2025-09-20               158    dados_aedes_20250920_weekid455_158mosquitos.xlsx
     456 2025-09-27               193    dados_aedes_20250927_weekid456_193mosquitos.xlsx
     457 2025-10-04               157    dados_aedes_20251004_weekid457_157mosquitos.xlsx
     459 2025-10-16               306    dados_aedes_20251016_weekid459_306mosquitos.xlsx
     460 2025

In [10]:
# ── Lê os arquivos selecionados e consolida em um único DataFrame ─────────────
dfs = []
for _, row in melhor.iterrows():
    df = pd.read_excel(row["arquivo"])
    # Garante que week_id no dado bate com o do nome do arquivo
    df["week_id"] = row["week_id"]
    df["data_raspagem"] = row["data"]
    dfs.append(df)

consolidado = pd.concat(dfs, ignore_index=True)

print(f"Registros totais consolidados: {len(consolidado):,}")
print(f"Week IDs cobertas: {consolidado['week_id'].nunique()}")
print(f"\nDistribuição de registros por week_id (amostra):")
print(consolidado.groupby("week_id")["total_mosquitos"].sum().to_string())
consolidado.head()

Registros totais consolidados: 28,210
Week IDs cobertas: 31

Distribuição de registros por week_id (amostra):
week_id
450      62
451      42
452      92
453      45
454      74
455     158
456     193
457     157
459     306
460     203
461     398
462     330
463     454
464     463
465     605
466     883
467     921
468     766
469     490
470     630
471    1366
472    1402
473    1135
474    1082
475    1015
476     703
477    1196
478    1209
479    1187
480    1228
481    1133


,id,week_id,trap_id,agent_id,week,agent,status,status_id,inspection_id,aedes_aegypti_femea,...,culex_sp_macho,total_mosquitos,latitude,longitude,street,number,block,complement,neighborhood,data_raspagem
0,4299153,450,11096,1065,33/2025,Lucas Bonesso,Realizada,10.0,1885818.0,0,...,0,0,-30.069557,-51.184382,AV SILVADO,456,NaN,ESCOLA TIO CHICO,APARICIO BORGES,2025-08-16
1,4299154,450,11097,1065,33/2025,Lucas Bonesso,Realizada,10.0,1885884.0,0,...,0,0,-30.068380,-51.180694,AV VEIGA,633,2237369.0,Cond. Jardim Partenon,APARICIO BORGES,2025-08-16
2,4299157,450,11100,1065,33/2025,Lucas Bonesso,Realizada,10.0,1885938.0,0,...,0,0,-30.073037,-51.177922,TRAV NOVA PRATA,46,8006865.0,Casa,APARICIO BORGES,2025-08-16
3,4299158,450,11104,1065,33/2025,Lucas Bonesso,Realizada,10.0,1885851.0,0,...,0,0,-30.073823,-51.181774,R SAO JORGE-CEL APARICIO BORGES,5,NaN,Casa,APARICIO BORGES,2025-08-16
4,4299160,450,11106,1065,33/2025,Lucas Bonesso,Realizada,10.0,1885844.0,0,...,0,0,-30.075200,-51.184420,R MUSICO NELSON FERREIRA,141,2237288.0,NaN,APARICIO BORGES,2025-08-16


In [11]:
# ── Exporta CSV final ─────────────────────────────────────────────────────────
consolidado.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"CSV salvo em: {OUTPUT_CSV}")
print(f"  Linhas: {len(consolidado):,}")
print(f"  Colunas: {list(consolidado.columns)}")

CSV salvo em: dados_tratados\dados_raspagem_consolidado.csv
  Linhas: 28,210
  Colunas: ['id', 'week_id', 'trap_id', 'agent_id', 'week', 'agent', 'status', 'status_id', 'inspection_id', 'aedes_aegypti_femea', 'aedes_aegypti_macho', 'aedes_albopictus_femea', 'aedes_albopictus_macho', 'culex_sp_femea', 'culex_sp_macho', 'total_mosquitos', 'latitude', 'longitude', 'street', 'number', 'block', 'complement', 'neighborhood', 'data_raspagem']
